# HMM Class

In [7]:
from scipy.stats import multivariate_normal
import numpy as np
from sklearn.cluster import KMeans

In [ ]:
class HMM:
    def __init__(self, K, X):
        # Use K-means to get smarter initial parameters than random
        kmeans = KMeans(n_clusters=K).fit(X)
        labels = kmeans.labels_

        self.X = X
        self.K = K

        # mu[k] — mean observation vector for regime k
        self.mu    = np.array([X[labels == k].mean(axis=0) for k in range(K)])

        # Sigma[k] — observation covariance matrix for regime k
        self.Sigma = np.array([np.cov(X[labels == k].T) for k in range(K)])

        # pi[k] — probability of starting in regime k
        self.pi    = np.array([(labels == k).mean() for k in range(K)])

        # A[j, k] — probability of transitioning from regime j to regime k
        self.A     = np.full((K, K), 1 / K)

    def _forward_pass(self):
        T     = len(self.X)
        alpha = np.zeros((T, self.K))

        # alpha[t, k] = P(o_1...o_t, s_t=k)
        # initialise at t=0 using starting probabilities
        for k in range(self.K):
            alpha[0, k] = (
                self.pi[k]
                * multivariate_normal.pdf(self.X[0], mean=self.mu[k], cov=self.Sigma[k])
            )

        # recurse forward — for each regime k, sum over all possible previous regimes j
        for t in range(1, T):
            for k in range(self.K):
                alpha[t, k] = (
                    multivariate_normal.pdf(self.X[t], mean=self.mu[k], cov=self.Sigma[k])
                    * np.sum(alpha[t - 1, :] * self.A[:, k])
                )

        return alpha  # shape (T, K)

    def _backward_pass(self):
        T    = len(self.X)
        beta = np.zeros((T, self.K))

        # beta[t, k] = P(o_{t+1}...o_T | s_t=k)
        # initialise at last timestep — no future observations to explain
        beta[T - 1, :] = 1

        # recurse backward — for each regime k, sum over all possible next regimes j
        for t in range(T - 2, -1, -1):
            for k in range(self.K):
                for j in range(self.K):
                    beta[t, k] += (
                        self.A[k, j]
                        * multivariate_normal.pdf(self.X[t + 1], mean=self.mu[j], cov=self.Sigma[j])
                        * beta[t + 1, j]
                    )

        return beta  # shape (T, K)

    def _compute_emission(self):
        T        = len(self.X)
        emission = np.zeros((T, self.K))

        # emission[t, k] = P(o_t | s_t=k) — precomputed to avoid redundant calls
        for t in range(T):
            for k in range(self.K):
                emission[t, k] = multivariate_normal.pdf(
                    self.X[t], mean=self.mu[k], cov=self.Sigma[k]
                )

        return emission  # shape (T, K)

    def _compute_posteriors(self, alpha, beta, emission):
        T = len(self.X)

        # gamma[t, k] = P(s_t=k | o_{1:T}) — soft regime assignment at each timestep
        gamma = np.zeros((T, self.K))
        for t in range(T):
            for k in range(self.K):
                gamma[t, k] = alpha[t, k] * beta[t, k]
            gamma[t] /= gamma[t].sum()  # normalise across regimes

        # xi[t, j, k] = P(s_t=j, s_{t+1}=k | o_{1:T}) — pairwise regime posterior
        # only defined for t = 0...T-2 since it involves adjacent timesteps
        xi = np.zeros((T - 1, self.K, self.K))
        for t in range(T - 1):
            for j in range(self.K):
                for k in range(self.K):
                    xi[t, j, k] = (
                        alpha[t, j]
                        * self.A[j, k]
                        * emission[t + 1, k]
                        * beta[t + 1, k]
                    )
            xi[t] /= xi[t].sum()  # normalise the KxK matrix at each timestep

        return gamma, xi  # shapes (T, K) and (T-1, K, K)

    def _m_step(self, gamma, xi):
        # Update pi — posterior regime probability at t=0
        self.pi = gamma[0]

        # Update A — expected transitions divided by expected time in each regime
        self.A = xi.sum(axis=0) / gamma[:-1].sum(axis=0, keepdims=True).T

        # Update mu — weighted mean of observations, weighted by regime probability
        for k in range(self.K):
            self.mu[k] = (
                (gamma[:, k:k+1] * self.X).sum(axis=0) / gamma[:, k].sum()
            )

        # Update Sigma — weighted covariance around updated mu
        for k in range(self.K):
            diff          = self.X - self.mu[k]
            weighted_diff = gamma[:, k:k+1] * diff
            self.Sigma[k] = (weighted_diff.T @ diff) / gamma[:, k].sum()

    def _log_likelihood(self, alpha):
        # Sum alpha across regimes at each t to get P(o_t | o_{1:t-1}), then sum logs
        return np.sum(np.log(alpha.sum(axis=1)))

    def _fit(self, tol=1e-6, max_iter=100):
        ll_prev = -np.inf

        for i in range(max_iter):
            # E-step — compute posteriors using current parameters
            alpha    = self._forward_pass()
            beta     = self._backward_pass()
            emission = self._compute_emission()
            gamma, xi = self._compute_posteriors(alpha, beta, emission)

            # Check convergence before updating parameters
            ll = self._log_likelihood(alpha)
            if abs(ll - ll_prev) < tol:
                print(f"Converged at iteration {i + 1}")
                break
            ll_prev = ll

            # M-step — update all parameters in closed form
            self._m_step(gamma, xi)
        else:
            print(f"Did not converge within {max_iter} iterations")

    def _viterbi(self):
        T     = len(self.X)
        delta = np.zeros((T, self.K))
        psi   = np.zeros((T, self.K), dtype=int)

        # delta[t, k] = probability of the most likely path ending in regime k at t
        # psi[t, k]   = which regime at t-1 led to that most likely path
        for k in range(self.K):
            delta[0, k] = (
                self.pi[k]
                * multivariate_normal.pdf(self.X[0], mean=self.mu[k], cov=self.Sigma[k])
            )
            psi[0, k] = 0

        for t in range(1, T):
            for k in range(self.K):
                candidates  = delta[t - 1, :] * self.A[:, k]
                delta[t, k] = (
                    multivariate_normal.pdf(self.X[t], mean=self.mu[k], cov=self.Sigma[k])
                    * candidates.max()
                )
                psi[t, k] = candidates.argmax()

        # backtrack from the best final regime to recover the full state sequence
        states      = np.zeros(T, dtype=int)
        states[T-1] = delta[T - 1, :].argmax()
        for t in range(T - 2, -1, -1):
            states[t] = psi[t + 1, states[t + 1]]

        return states  # shape (T,) — hard regime label per timestep

    def train(self):
        # Step 1 — fit parameters via Baum-Welch EM
        self._fit()

        # Step 2 — final forward/backward with fitted parameters
        alpha    = self._forward_pass()
        beta     = self._backward_pass()
        emission = self._compute_emission()

        # Step 3 — soft regime probabilities (gamma) for XGBoost and LSTM input
        gamma, _ = self._compute_posteriors(alpha, beta, emission)

        # Step 4 — hard regime sequence via Viterbi for transition row lookup
        states = self._viterbi()

        # Step 5 — transition row A_k[t] is the row of A for the regime at time t
        # tells you how likely each regime is tomorrow given today's regime
        A_k = self.A[states]  # shape (T, K)

        # Step 6 — 5-day crisis transition probability for position sizing damping
        # crisis regime = the one with highest total variance (largest trace of Sigma)
        A_5            = np.linalg.matrix_power(self.A, 5)
        crisis_regimes = [np.argmax([self.Sigma[k].trace() for k in range(self.K)])]
        crisis_prob    = np.array([
            A_5[states[t], crisis_regimes].sum()
            for t in range(len(self.X))
        ])  # shape (T,)

        return (
            gamma,       # shape (T, K)  — soft regime probabilities
            states,      # shape (T,)    — hard regime label per day
            A_k,         # shape (T, K)  — transition row for current regime
            crisis_prob  # shape (T,)    — 5-day crisis probability
        )
    
    def predict(self, X_new):
        # Run forward pass only — at inference time future observations are not available
        # so we cannot use the backward pass to compute the full posterior
        T     = len(X_new)
        alpha = np.zeros((T, self.K))

        # Initialise at t=0
        for k in range(self.K):
            alpha[0, k] = (
                self.pi[k]
                * multivariate_normal.pdf(X_new[0], mean=self.mu[k], cov=self.Sigma[k])
            )

        # Forward recurse using fitted parameters
        for t in range(1, T):
            for k in range(self.K):
                alpha[t, k] = (
                    multivariate_normal.pdf(X_new[t], mean=self.mu[k], cov=self.Sigma[k])
                    * np.sum(alpha[t - 1, :] * self.A[:, k])
                )

        # Normalise each row to get filtered regime probabilities
        gamma = alpha / alpha.sum(axis=1, keepdims=True)  # shape (T, K)

        # Hard regime labels from filtered probabilities
        states = gamma.argmax(axis=1)  # shape (T,)

        # Transition row for each timestep
        A_k = self.A[states]  # shape (T, K)

        # 5-day crisis probability
        A_5            = np.linalg.matrix_power(self.A, 5)
        crisis_regimes = [np.argmax([self.Sigma[k].trace() for k in range(self.K)])]
        crisis_prob    = np.array([
            A_5[states[t], crisis_regimes].sum()
            for t in range(T)
        ])  # shape (T,)

        return (
            gamma,       # shape (T, K)  — soft regime probabilities
            states,      # shape (T,)    — hard regime label per day
            A_k,         # shape (T, K)  — transition row for current regime
            crisis_prob  # shape (T,)    — 5-day crisis probability
        )